In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
# 改了 src/ 里的 .py，notebook 不用重启就能用上新代码。
from signal_mysql import database as mysql
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, geo, privacy, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT    # 配色常量
viz.apply_theme()                           # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)       # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY                   # 分析日期（口径常量）
NB = "D07_出行与过夜地"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"    # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
from signal_mysql import database as mysql
# 【跟丢了才运行】D6 没做完的同学：把 False 改成 True 再运行，从检查点库复制标准表
CATCH_UP = False
if CATCH_UP:
    with mysql.connect(config.CHECKPOINT_SCHEMA) as ck:
        for t in ["dwd_signal", "dws_stay", "dim_hotspot"]:
            mysql.write_frame(mysql.read_frame(f"SELECT * FROM {t}", ck), t, con, if_exists="replace", index=False)
            print("已恢复", t)

In [ ]:
from signal_mysql import database as mysql
from signal_insight import db, dws, pipeline, privacy
settings = config.load_settings()
hot = mysql.read_frame("SELECT * FROM dim_hotspot", con).iloc[0]       # 西塔中心（D4）
DAY_START = pd.Timestamp(DAY)                                      # 2018-10-03 00:00
DISTRICTS = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_district_boundary", con), "district")
AOIS = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_aoi_v2", con), "name")

In [ ]:
from signal_mysql import database as mysql
stay = mysql.read_frame("SELECT * FROM dws_stay", con, parse_dates=["start_time", "end_time"])
n_users = mysql.read_frame("SELECT COUNT(DISTINCT imsi) AS n FROM dwd_signal", con)["n"][0]
print(f"停留 {len(stay):,} 个，全部用户 {n_users:,} 人")

In [ ]:
# ════ 本日第一个关键概念：过夜地 ════
# 这一格在做什么：找出每个人"凌晨 0–6 点待得最久的那个停留"，作为他的过夜地。
#
# 两个区间重叠多少分钟，标准算法就是下面这一行：
#   重叠 = min(结束时刻, 区间右端) − max(开始时刻, 区间左端)，算出来是负数就当 0。
# clip(upper=...) 和 clip(lower=...) 正是在做 min 和 max，最后 clip(lower=0) 处理"完全不重叠"。
#
# ⚠ **过夜地 ≠ 家**。它只是一个代理指标：国庆期间住酒店的人，过夜地就是酒店；
# 通宵在外的人，过夜地可能是个饭馆。所有用到它的结论都要写清楚这一点，
# 口径表 dim_metric 里也特意写成"凌晨待得最久的地方（不等于家）"。
#
# 为什么要求重叠 ≥ 60 分钟：十几分钟的停留没法代表"过夜"，宁可认成"识别不出"。
# 这个阈值和 500 米、30 分钟一样写在配置里（night.min_overlap_minutes），不是硬编码。
night_end = DAY_START + pd.Timedelta(hours=6)
overlap = stay["end_time"].clip(upper=night_end) - stay["start_time"].clip(lower=DAY_START)
stay["night_min"] = (overlap.dt.total_seconds() / 60).clip(lower=0)             # 与 0–6 点重叠的分钟数
night = (stay[stay["night_min"] >= 60]
          .sort_values("night_min")
          .groupby("imsi").tail(1)                                             # 排序后取每组最后一条 = 重叠最长的
          .set_index("imsi"))
# ⚠ **识别率要一直带着走**：只有一部分人能识别出过夜地（信令太稀疏的人识别不出）。
# 后面每一个基于 night 的比例，分母都是这些人，不是全部用户。
# 所以图上和报告里必须写成"能识别过夜地的访客（占 XX%）中……"，
# 不能说成"XX% 的访客住在……"。这是 D8、D9、D10 反复检查的一条。
print(f"能识别过夜地的用户：{len(night):,} 人，占 {len(night) / n_users:.1%}")
night[["start_time", "end_time", "night_min", "district", "dist_xita_m"]].head(3).rename(index=privacy.mask_id)    # 展示前脱敏

In [ ]:
# 对账
assert 3_590 <= len(night) <= 3_620
print(f"✅ 对账通过：{len(night):,} 人能识别过夜地（约 64%）")

In [ ]:
# ── 出图 D7_01：六边形热力地图 + 条形图 ───────────────────────────────
# 为什么用六边形而不是散点：几千个过夜地画成散点会糊成一团黑，看不出密度差异。
# 为什么是六边形而不是方格：六边形的六个邻居到中心距离相同（方格的对角邻居更远），
# 做空间聚合时不会出现"沿着格子边界的方向性伪影"。
# 规范落在哪几行：
#   · norm=LogNorm(5, 300) —— 人数跨了两个数量级，线性色阶会让除了最热的几格以外全是一个色；
#     色条刻度写成 5 / 10 / 30 / 100 / 300 这样的**实际数值**，不写 10¹、10²
#   · mincnt=5 —— **小样本抑制**：少于 5 人的格子直接不画。这是隐私底线，
#     而且副标题里写明了"显示的格子覆盖 XX% 的人"，让读者知道被抹掉了多少
#   · BLUES 单色渐变 —— 连续数值不用彩虹色阶
#   · set_aspect(1 / cos(41.8°)) —— 地图长宽比校正
# 标题里的数字：top_d["和平区"]，即过夜地落在和平区的人数占比。
from matplotlib.colors import LinearSegmentedColormap, LogNorm
x0, x1, y0, y1 = 123.18, 123.70, 41.64, 41.98                         # 沈阳主城区范围
top_d = night["district"].value_counts(normalize=True)
BLUES = LinearSegmentedColormap.from_list("blues", ["#EAF2F9", MAIN_MID, MAIN, MAIN_DARK])    # 单色渐变，不用彩虹色阶
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), layout="constrained", width_ratios=[1.5, 1])
viz.maps.draw_districts(a1, DISTRICTS, edge="#8E99A6")
v_ = night[night["lng"].between(x0, x1) & night["lat"].between(y0, y1)]
hb = a1.hexbin(v_["lng"], v_["lat"], gridsize=(46, 23), extent=(x0, x1, y0, y1), mincnt=5,    # 少于 5 人的格子不显示
               cmap=BLUES, norm=LogNorm(5, 300), lw=0.2, edgecolors="white", zorder=3)
shown = hb.get_array().sum() / len(night)
a1.set_xlim(x0, x1); a1.set_ylim(y0, y1)
viz.maps.set_map_aspect(a1)           # 长宽比校正 + 去掉刻度、网格、边框
t_ = np.linspace(0, 2 * np.pi, 100)
a1.plot(hot["lng"] + 0.5 / 111.32 / np.cos(np.radians(41.8)) * np.cos(t_),
        hot["lat"] + 0.5 / 111.32 * np.sin(t_),
        color=INK, lw=1.2, ls="--", zorder=5)                           # 西塔 500 米
a1.annotate("西塔", xy=(hot["lng"], hot["lat"]), xytext=(hot["lng"] + 0.05, hot["lat"] + 0.045), fontsize=10,
            fontweight="bold", color=INK, arrowprops=dict(arrowstyle="-", color=INK, lw=0.8), zorder=6)
viz.maps.scale_bar(a1, 5); viz.maps.north_arrow(a1)
cb = fig.colorbar(hb, ax=a1, shrink=0.6, pad=0.01, location="left")
cb.set_ticks([5, 10, 30, 100, 300], labels=["5", "10", "30", "100", "300"]); cb.outline.set_visible(False)
cb.set_label("过夜人数（对数色阶）", color=MUTED)
d7 = top_d.head(7)[::-1]
bars = a2.barh(d7.index, d7.values, color=[MAIN if k == "和平区" else GRAY for k in d7.index], height=0.62)
a2.bar_label(bars, labels=[f"{v:.0%}" for v in d7.values], padding=3, fontsize=9.5, color=INK)
a2.set_xlim(0, d7.max() * 1.3); a2.set_xticks([]); a2.grid(False); a2.tick_params(axis="y", labelcolor=INK)
a2.set_title("过夜地所在区县（前 7）", loc="left", fontsize=10.5, color=INK, fontweight="bold")
title = f"过夜地集中在西塔周边，和平区占 {top_d['和平区']:.0%}"
viz.finish(fig, title,
           f"左：每个六边形内的过夜人数，少于 5 人的格子不显示（显示的格子覆盖 {shown:.0%} 的人），只画主城区；\n过夜地 = 与 0–6 点重叠最长且 ≥60 分钟的停留；共 {len(night):,} 人",
           SOURCE, "D7_01_过夜地分布.png")

In [ ]:
# ── 出图 D7_02：ECDF，距离带作背景 ──────────────────────────────────
# 为什么 ECDF 配背景色带：客户关心的是"多少人住在 1.5 公里内"这种累计问题（ECDF 直接读），
# 但他习惯按"近 / 中 / 远"几档来说话（色带给分档）。两者叠在一张图上，各取所长。
# 规范落在哪几行：
#   · 虚线 + 圆点标出中位数，旁边写明数值 —— 关键分位数要直接标在图上
#   · 副标题写"直线距离，不是路程" —— 口径限制。实际走的路一定比直线长
#   · 副标题写"能识别过夜地的 N 人" —— 识别率，分母说清楚
# 标题里的数字：med_km（距离中位数）、band_share.iloc[-1]（最远那一档的占比）。
bands = pd.cut(night["dist_xita_m"] / 1000, bins=[0, 1.5, 5, 15, np.inf], right=False,
               labels=["1.5 公里内", "1.5–5 公里", "5–15 公里", "15 公里以上"])
band_share = bands.value_counts(normalize=True).sort_index()
mysql.write_frame(band_share.rename("share").rename_axis("distance_band").reset_index(), "ads_night_distance", con, if_exists="replace", index=False)
med_km = night["dist_xita_m"].median() / 1000
km = night["dist_xita_m"] / 1000
fig, ax = plt.subplots(figsize=(10, 5.6), layout="constrained")
# ECDF（手册 5.5）标出中位数；原来的四个距离带留作背景
for (a, b), lab, c_ in zip([(0, 1.5), (1.5, 5), (5, 15), (15, 30)], band_share.index, [MAIN_LIGHT, "#EEF3F8", "#F6F8FA", "white"]):
    ax.axvspan(a, b, color=c_, lw=0, zorder=0)
    ax.text((a + b) / 2, 1.05, f"{lab}\n{band_share[lab]:.0%}", ha="center", fontsize=9, color=MUTED)
ax.ecdf(km, color=MAIN, lw=2.2)
ax.plot([med_km, med_km], [0, 0.5], color=INK, lw=1, ls="--"); ax.plot(med_km, 0.5, "o", color=MAIN, ms=6)
ax.text(med_km + 0.4, 0.52, f"中位数 {med_km:.1f} 公里", fontsize=10, color=INK, fontweight="bold")
ax.set_xlim(0, 30); ax.set_ylim(0, 1.16); ax.set_yticks(np.arange(0, 1.01, 0.2))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("过夜地到西塔中心的直线距离（公里）"); ax.set_ylabel("距离不超过横轴数值的人"); ax.grid(axis="both")
title = f"一半的人过夜地离西塔不到 {med_km:.1f} 公里，{band_share.iloc[-1]:.0%} 在 15 公里以外"
viz.finish(fig, title,
           f"能识别过夜地的 {len(night):,} 人；累计分布：曲线上一点 = 距离不超过横轴数值的人数占比；直线距离，不是路程；口径：dws_night",
           SOURCE, "D7_02_过夜地距离.png")

In [ ]:
# 对账
assert 2.5 < med_km < 3.2 and 0.05 < band_share.iloc[-1] < 0.07
print(f"✅ 对账通过：过夜地离西塔中位 {med_km:.1f} 公里")

In [ ]:
# ════ 本日第二个关键概念：出行 ════
# 这一格在做什么：把每个人相邻的两个停留连起来，就是一次"出行"。
#   起点 = 上一个停留的质心和**结束**时刻；终点 = 下一个停留的质心和**开始**时刻。
#
# 注意我们并不知道他路上怎么走的——信令只告诉我们"他在这停过、又在那停过"。
# 所以 dws_trip 里的 dist_m 是**直线距离**，duration_min 是**两次停留之间的全部时间**
# （包含了在路边等车、吃饭这些没形成停留的时间）。7.6 用这两个数算速度，
# 结论正是"算出来偏低，没法可靠判断交通方式"——这是诚实的限制，不是算错了。
s = stay.sort_values(["imsi", "stay_seq"])
nxt = s.groupby("imsi").shift(-1)                                    # 同一个人的下一个停留
trip = pd.DataFrame({
    "imsi": s["imsi"], "o_seq": s["stay_seq"], "d_seq": nxt["stay_seq"],
    "depart_time": s["end_time"], "arrive_time": nxt["start_time"],
    "o_lng": s["lng"], "o_lat": s["lat"], "d_lng": nxt["lng"], "d_lat": nxt["lat"],
    "o_district": s["district"], "d_district": nxt["district"],
    "o_aoi": s["aoi"], "d_aoi": nxt["aoi"],
    "o_dist_xita_m": s["dist_xita_m"], "d_dist_xita_m": nxt["dist_xita_m"],
}).dropna(subset=["d_seq"])                                          # 最后一个停留后面没有出行
trip["dist_m"] = geo.haversine(trip["o_lat"], trip["o_lng"], trip["d_lat"], trip["d_lng"]).round()
# 为什么要求相距 ≥ 500 米才算一次出行：两个停留的质心相距两三百米，
# 更可能是同一个地方被切成了两段（人下楼买了杯咖啡又回去），不是真的"出了一趟门"。
# 500 米这个阈值同样写在配置里（trip.min_dist_m）。
# ⚠ 又一个"口径决定数字"的例子：阈值定成 200 米，出行次数会多出一大截。
trip = trip[trip["dist_m"] >= 500].copy()
trip["duration_min"] = ((trip["arrive_time"] - trip["depart_time"]).dt.total_seconds() / 60).round(1)
trip["trip_seq"] = trip.groupby("imsi").cumcount() + 1
print(f"出行 {len(trip):,} 次，涉及 {trip['imsi'].nunique():,} 人；距离中位 {trip['dist_m'].median() / 1000:.1f} 公里")

In [ ]:
from signal_mysql import database as mysql
# 这一格在做什么：筛出"傍晚从外面来到西塔"的那些出行，看他们从哪儿出发。
# 三个条件缺一不可：
#   d_dist_xita_m <= 500    终点在西塔范围内
#   o_dist_xita_m > 500     起点在**外面**（不写这条，西塔内部的挪动也会被算进来）
#   arrive_time.hour >= 17  傍晚到达
# 讲台提示：让学生试试去掉第二个条件，看数字变多少——这是最容易漏掉的一条。
to_xita = trip[(trip["d_dist_xita_m"] <= 500) & (trip["o_dist_xita_m"] > 500) & (trip["arrive_time"].dt.hour >= 17)]
origin = to_xita["o_district"].value_counts(normalize=True).rename("share").rename_axis("origin_district").reset_index()
mysql.write_frame(origin, "ads_xita_origin", con, if_exists="replace", index=False)
o_km = to_xita["dist_m"].median() / 1000
print(f"傍晚到达西塔的出行 {len(to_xita):,} 次（{to_xita['imsi'].nunique():,} 人）；出发地离西塔中位 {o_km:.1f} 公里")
origin.head(6)

In [ ]:
# ── 出图 D7_03：分级设色地图 + 条形图 ───────────────────────────────
# 为什么这次用"按区县填色"而不是六边形热力：这里的统计单位本来就是**区县**
# （我们算的是"从哪个区出发"），填色的边界和统计的边界一致，读者不会误读。
# D7_01 那张统计的是连续分布的点位，才用六边形。
#    **统计单位是行政区 → 分级设色；统计单位是点位 → 六边形热力。**
# 规范落在哪几行：
#   · Normalize(0, sh.max()) 而不是 (0, 1) —— 最大值才 30% 左右，用 0–1 的话全图都是浅色
#   · 只给四个主要区县加 annotate 标注 —— 不是每个区都标，避免文字糊成一片
#   · set_aspect(1 / cos(41.8°)) —— 地图长宽比校正
# 标题里的数字：o["share"].iloc[0] 和 o["origin_district"].iloc[0]（占比最高的出发地），
# 以及 o_km（出发地到西塔的距离中位数）。
from matplotlib.colors import LinearSegmentedColormap, Normalize
o = mysql.read_frame("SELECT * FROM ads_xita_origin", con)
sh = to_xita["o_district"].value_counts(normalize=True)
x0, x1, y0, y1 = 123.18, 123.70, 41.64, 41.98                         # 主城区
BLUES = LinearSegmentedColormap.from_list("blues", ["#EAF2F9", MAIN_MID, MAIN, MAIN_DARK])    # 单色渐变，不用彩虹色阶
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), layout="constrained", width_ratios=[1.3, 1])
norm = Normalize(0, sh.max())
viz.maps.draw_districts(a1, DISTRICTS, values=sh.to_dict(), cmap=BLUES, norm=norm, edge="white", lw=0.9)
a1.set_xlim(x0, x1); a1.set_ylim(y0, y1)
viz.maps.set_map_aspect(a1)           # 长宽比校正 + 去掉刻度、网格、边框
for dname, (dx, dy) in {"和平区": (-0.03, -0.11), "皇姑区": (-0.02, 0.06), "铁西区": (-0.12, -0.03), "沈河区": (0.10, -0.02)}.items():
    gx, gy = np.vstack(DISTRICTS[dname]).mean(axis=0)                       # 区县所有顶点的平均位置
    a1.annotate(f"{dname} {sh.get(dname, 0):.0%}", xy=(gx, gy),
                xytext=(gx + dx, gy + dy), fontsize=9, color=INK, fontweight="bold", ha="center",
                arrowprops=dict(arrowstyle="-", color=INK, lw=0.7), zorder=6)
a1.plot(hot["lng"], hot["lat"], marker="*", ms=11, color=INK, zorder=7)
viz.maps.scale_bar(a1, 5); viz.maps.north_arrow(a1)
o_top = o.head(8).iloc[::-1]
bars = a2.barh(o_top["origin_district"], o_top["share"], color=[MAIN if i == len(o_top) - 1 else GRAY for i in range(len(o_top))], height=0.64)
a2.bar_label(bars, labels=[f"{v:.0%}" for v in o_top["share"]], padding=3, fontsize=9.5, color=INK)
a2.set_xlim(0, o_top["share"].max() * 1.25); a2.set_xticks([]); a2.grid(False); a2.tick_params(axis="y", labelcolor=INK)
a2.set_title("前 8 个出发区县", loc="left", fontsize=10.5, color=INK, fontweight="bold")
title = f"傍晚到达西塔的出行，{o['share'].iloc[0]:.0%} 从{o['origin_district'].iloc[0]}出发；出发地离西塔中位 {o_km:.1f} 公里"
viz.finish(fig, title,
           f"17 点后从西塔 500 米外到达 500 米内的 {len(to_xita):,} 次出行，按出发地所在区县统计；左图只画主城区，★ = 西塔；口径：dws_trip",
           SOURCE, "D7_03_傍晚到达西塔的出发地.png")

In [ ]:
# 收进包的版本和上面一步步算的结果对账，然后写回库里
dws_night = dws.build_night(stay, DAY, settings)
dws_trip = dws.build_trip(stay, settings)
print(f"过夜地：包 {len(dws_night):,} 人 / notebook {len(night):,} 人；出行：包 {len(dws_trip):,} 次 / notebook {len(trip):,} 次")
db.replace_table(con, dws_night, "dws_night")
db.replace_table(con, dws_trip, "dws_trip")
# 对账：与 notebook 版本一致
assert len(dws_night) == len(night) == 3_607 and len(dws_trip) == len(trip) == 11_055
assert set(dws_night["imsi"]) == set(night.index)
print("✅ 对账通过：过夜地 3,607 人、出行 11,055 次，与 notebook 版本一致")

In [ ]:
# 流水线多了两步：dws_night、dws_trip（整表替换）
log = pipeline.run_day()
log[["step", "rows_in", "rows_out", "elapsed_sec"]].tail(6)

In [ ]:
# 这一格在做什么：用出行距离 ÷ 时长算平均速度，看能不能判断交通方式。
# 结论是**不能**，而且这个"做不到"本身就是交付的结论之一。原因有两条，都写在了图的副标题里：
#   · 分子偏小：直线距离短于实际路程；
#   · 分母偏大：出发和到达时刻来自断续的信令，中间等车、吃饭的时间也算进去了。
# 两头一挤，算出来的速度系统性偏低。
# 讲台提示：企业里经常被要求"顺便判断一下交通方式"。能说清"这份数据做不到，为什么做不到"，
# 比硬给一个不可靠的数字有价值得多。
v = to_xita[to_xita["duration_min"] > 0]
speed = (v["dist_m"] / 1000) / (v["duration_min"] / 60)
bands_v = pd.cut(speed, bins=[0, 6, 15, 30, np.inf], right=False,
                 labels=["6 公里/时以下", "6–15", "15–30", "30 公里/时以上"]).value_counts(normalize=True).sort_index()
fig, ax = plt.subplots(figsize=(10, 5.6))
bars = ax.bar(bands_v.index.astype(str), bands_v.values, color=GRAY, width=0.6)
ax.bar_label(bars, labels=[f"{x:.0%}" for x in bands_v.values], padding=4, color=INK)
ax.set_yticks([]); ax.grid(False)
ax.set_ylim(0, bands_v.max() * 1.18)
ax.set_xlabel("按信令估算的平均速度（直线距离 ÷ 两次停留之间的时间）", color=MUTED)
title = f"按信令估算，傍晚来西塔的出行中位速度只有 {speed.median():.0f} 公里/时，无法可靠判断交通方式"
viz.finish(fig, title,
           "出发、到达时刻来自断续的信令，时长偏长、速度偏低；直线距离短于实际路程；口径：dws_trip",
           SOURCE, "D7_04_出行速度.png")

In [ ]:
from signal_mysql import database as mysql
metro = mysql.read_frame("SELECT name, line, lng, lat FROM ods_poi_transport WHERE type = '地铁'", con)
metro["到西塔(米)"] = geo.haversine(metro["lat"], metro["lng"], hot["lat"], hot["lng"]).round()
metro.nsmallest(3, "到西塔(米)")[["name", "line", "到西塔(米)"]]

In [ ]:
from signal_mysql import database as mysql
trip_all = mysql.read_frame("SELECT * FROM dws_trip", con, parse_dates=["depart_time", "arrive_time"])
main = ["和平区", "沈河区", "皇姑区", "铁西区", "大东区", "浑南区", "于洪区", "沈北新区", "苏家屯区"]
order = main + ["郊县及市外"]
o = trip_all["o_district"].where(trip_all["o_district"].isin(main), "郊县及市外")
d = trip_all["d_district"].where(trip_all["d_district"].isin(main), "郊县及市外")
od = pd.crosstab(o, d).reindex(index=order, columns=order, fill_value=0)
cross = trip_all["o_district"] != trip_all["d_district"]
a = np.where(trip_all["o_district"] < trip_all["d_district"], trip_all["o_district"], trip_all["d_district"])
b = np.where(trip_all["o_district"] < trip_all["d_district"], trip_all["d_district"], trip_all["o_district"])
pairs = pd.Series([f"{x}↔{y}" for x, y in zip(a[cross], b[cross])]).value_counts()    # 不分方向的区间往来
# 往返：这次出行的起点，和下一次出行的终点相距 500 米内（A → B → A）
t = trip_all.sort_values(["imsi", "trip_seq"])
nxt = t.groupby("imsi")[["d_lng", "d_lat"]].shift(-1)
aba = pd.Series(geo.haversine(t["o_lat"], t["o_lng"], nxt["d_lat"], nxt["d_lng"]) <= 500, index=t.index)
hh = t["depart_time"].dt.hour
print(f"跨区出行占 {cross.mean():.0%}；凌晨 0–5 点出行 {(hh < 5).sum()} 次，其中往返 {aba[hh < 5].mean():.1%}；白天 8–20 点往返 {aba[(hh >= 8) & (hh < 20)].mean():.1%}")
pairs.head(5)

In [ ]:
# 对账
assert len(trip_all) == 11_055 and round(cross.mean(), 2) == 0.64 and (hh < 5).sum() == 770
print("✅ 对账通过：11,055 次出行，跨区 64%，凌晨 770 次")

In [ ]:
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MPath
# ── 出图 D7_05：流向弧线地图 + 条形图 ───────────────────────────────
# 为什么用弧线不用直线：两个区之间来回都有流量，直线会完全重叠成一条。
# 弧线往一侧弯开（下面 cx, cy 那两行就是把控制点推到中垂线上），两个方向各走一边。
# 为什么**不用和弦图**：和弦图好看但读不准——圆周上的弧长没法目测比较。
# 右边配一张排序条形图，才能回答"到底哪两对最多、差多少"。
# 规范落在哪几行：
#   · p = p[p >= 5] —— 少于 5 次的区县对不画（小样本抑制）
#   · lw = 0.6 + 7 * v / p.max() —— 线宽正比于流量，并且给一个最小值，否则细线看不见
#   · 只有前 3 对上主色 —— 一种颜色强调一件事
#   · set_aspect(1 / cos(41.8°)) —— 地图长宽比校正
# 标题里的数字：cross.mean()（跨区出行的比例）、p.index[0] 与 p.iloc[0]（往来最多的那一对）。
def flow_map(ax, od, x0, x1, y0, y1, top_n=3):
    """流向弧线地图：od 有 o、d、trips 三列；跨区按无向合并，少于 5 次不画；区内出行画成白圈。"""
    viz.maps.draw_districts(ax, DISTRICTS, edge="#C4CAD3", lw=0.7)
    # 每个区县取顶点最多的那个环（主体，不是飞地）的平均位置，当弧线的端点
    cen = {name: tuple(max(rings, key=len).mean(axis=0)) for name, rings in DISTRICTS.items()}
    intra = od[od["o"] == od["d"]].set_index("o")["trips"]
    x_ = od[od["o"] != od["d"]].copy()
    x_["pair"] = [tuple(sorted(p)) for p in zip(x_["o"], x_["d"])]
    p = x_.groupby("pair")["trips"].sum().sort_values(ascending=False)
    p = p[p >= 5]
    p = p[[a in cen and b in cen for a, b in p.index]]
    for (a, b), v in p[::-1].items():
        (xa, ya), (xb, yb) = cen[a], cen[b]
        cx, cy = (xa + xb) / 2 - (yb - ya) * 0.18, (ya + yb) / 2 + (xb - xa) * 0.18
        hi = (a, b) in list(p.index[:top_n])
        ax.add_patch(PathPatch(MPath([(xa, ya), (cx, cy), (xb, yb)],
                                     [MPath.MOVETO, MPath.CURVE3, MPath.CURVE3]),
                               fill=False, lw=0.6 + 7 * v / p.max(), color=MAIN if hi else GRAY_DARK,
                               alpha=0.95 if hi else 0.45, capstyle="round", zorder=4 if hi else 3))
    for dname, v in intra.items():
        if dname in cen:
            ax.scatter(*cen[dname], s=v / intra.max() * 420, facecolor="white", edgecolor=GRAY_DARK, lw=1, zorder=5)
    for dname, (cx, cy) in cen.items():
        if x0 < cx < x1 and y0 < cy < y1:
            ax.text(cx, cy - 0.016, dname.replace("区", ""), ha="center", va="top", fontsize=8.8, color=INK, zorder=6)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    return p
od_ = trip_all.groupby(["o_district", "d_district"]).size().rename("trips").reset_index()
od_.columns = ["o", "d", "trips"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), layout="constrained", width_ratios=[1.45, 1])
p = flow_map(a1, od_, 123.18, 123.70, 41.64, 41.98)
viz.maps.set_map_aspect(a1)           # 长宽比校正 + 去掉刻度、网格、边框
viz.maps.scale_bar(a1, 5); viz.maps.north_arrow(a1)
top8 = p.head(8)[::-1]
bars = a2.barh([f"{a.replace('区', '')} ↔ {b.replace('区', '')}" for a, b in top8.index], top8.values,
               color=[MAIN if i >= len(top8) - 3 else GRAY for i in range(len(top8))], height=0.64)
a2.bar_label(bars, labels=[f"{v:,}" for v in top8.values], padding=3, fontsize=9.5, color=INK)
a2.set_xlim(0, top8.max() * 1.22); a2.set_xticks([]); a2.grid(False); a2.tick_params(axis="y", labelcolor=INK)
a2.set_title("往返合计最多的 8 对区县（次）", loc="left", fontsize=10.5, color=INK, fontweight="bold")
title = f"{cross.mean():.0%} 的出行跨区，往来最多的是 {p.index[0][0]} 与 {p.index[0][1]}（{p.iloc[0]:,} 次）"
viz.finish(fig, title,
           f"{len(trip_all):,} 次出行（相邻停留点相距 ≥ 500 米）；线宽 = 两区往返合计，少于 5 次不画；白圈 = 区内出行；蓝线 = 前 3 对；口径：dws_trip",
           SOURCE, "D7_05_区县OD.png")

In [ ]:
con.close()